In [ ]:
#Notbook to run ISM for a given bed file in

%load_ext autoreload
%autoreload 2

import os
import sys
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy import stats

# Make the package in ../src importable without installing it.
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from element_epistasis import paths
from element_epistasis.ism import load_ism_dict

paths.ensure_output_dirs()

# True regenerates the AlphaGenome predictions, which needs an API key, the
# hg38 reference and several hours. False loads them from data/processed.
RUN_PREDICTIONS = False

import element_epistasis.tf_cooperativity as tc
import element_epistasis.tf_plots as tp

In [ ]:
%load_ext autoreload
%autoreload 2
import element_epistasis.tf_plots as tp
import element_epistasis.tf_cooperativity as tc
import element_epistasis.cross_level as cl

import importlib
importlib.reload(tc)
importlib.reload(tp)

In [ ]:
# The key is read from ALPHAGENOME_API_KEY or ~/.alphagenome_api_key.
api_key = paths.alphagenome_api_key()
model = dna_client.create(api_key) if api_key else None
if RUN_PREDICTIONS and model is None:
    raise RuntimeError("RUN_PREDICTIONS needs an AlphaGenome API key")


In [ ]:
enhancers = pd.read_csv(paths.RAW / "enhancers.bed", sep = "\t", header=None, names=["chrom", "start", "end", "name"])
def extract_genes(name_string):
    return [part.split('_')[0] for part in name_string.split(':')]
def extract_enhancers(name_string):
    return [part.split('_')[1] for part in name_string.split(':')]

enhancers['target_gene'] = enhancers['name'].apply(extract_genes)
enhancers['enhancer_nr'] = enhancers['name'].apply(extract_enhancers)

all_TFs = pd.read_csv(paths.RAW / "tfs.tsv", sep = "\t", header=None, names=["TF"])
all_TFs = all_TFs.drop_duplicates().TF.to_list()

enhancers

In [ ]:
if model is not None:
    # selecting relvant tracks from track metadata
    track_metadata = model.output_metadata(
        dna_client.Organism.HOMO_SAPIENS
    ).concatenate()

    K562_metadata = track_metadata[(track_metadata["biosample_name"].str.contains("K562", case=False, na=False))&
                   ((track_metadata["output_type"] == dna_client.OutputType.ATAC) | (track_metadata["output_type"] == dna_client.OutputType.CHIP_TF))
                  ].dropna(how="all", axis = 1)

    K562_curie = K562_metadata.ontology_curie.unique().tolist()
    K562_sample = K562_metadata.biosample_name.unique().tolist()

    print("curie:", K562_curie)
    print("samples:", K562_sample)
else:
    # No API key: use the curie the stored predictions were generated with.
    K562_curie = [paths.K562_CURIE]
    print("curie:", K562_curie, "(fallback, no API key)")


In [ ]:
if model is not None:
    TFs = [tf.upper() for tf in all_TFs if tf.upper() in K562_metadata.transcription_factor.str.upper().to_list()]


In [ ]:
scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['ATAC'], variant_scorers.RECOMMENDED_VARIANT_SCORERS['CHIP_TF'], variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]
SEQ_LEN = dna_client.SEQUENCE_LENGTH_16KB

In [ ]:
enhancers["ivs"] = [genome.Interval(chromosome=row.chrom, start=row.start, end=row.end) for row in enhancers.itertuples()]
enhancers["ivs_big"] = [iv.resize(SEQ_LEN) for iv in enhancers["ivs"]]

In [ ]:
ism_enhancer = load_ism_dict(paths.PROCESSED / "K562_ISM__enhancers.h5")

In [ ]:
# The GTF file contains information on the location of all trancripts.
# Note that we use genome assembly hg38 for human.
gtf = paths.load_gtf()

# Set up transcript extractors using the information in the GTF file.
# Mane select transcripts consists of of one curated transcript per locus.
#gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_mane_select_transcript(gtf)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

In [ ]:
def plot_ISM(res_dict, interval, variants = None, title = ""):  
    #plotting scores as logos
    seq_logo_data = [plot_components.SeqLogo(
                            scores=attributions,
                            scores_interval=interval,
                            ylabel= f"{m}",
                        ) for m, attributions in res_dict.items()
                    ]

    if variants:
        annot = [plot_components.VariantAnnotation(variants, alpha=0.15, colors="yellow", label_angle=50),
                    # plot_components.VariantAnnotation(sig_mutations, alpha=0.5, colors="red")
                    ]
    else: annot = None
    plot_components.plot(
        [plot_components.TranscriptAnnotation(transcripts = transcript_extractor.extract(interval)), *seq_logo_data],
        #seq_logo_data,
        title = f"{title}",
        interval=interval,
        fig_width=35,
        annotations=annot,
    )
    plt.show()

In [ ]:
name = "ALDH1A1_enh1"
plot_ISM(ism_enhancer[name], enhancers[enhancers.name == name].ivs.values[0])

In [ ]:
name = "ALDH1A1_enh2:ANXA1_enh1"
plot_ISM(ism_enhancer[name], enhancers[enhancers.name == name].ivs.values[0])

In [ ]:
name = "SLC25A37_enh1"
plot_ISM(ism_enhancer[name], enhancers[enhancers.name == name].ivs.values[0])

# Run TF-MODISCO to find motifs in the sequence

In [ ]:
import torch
from tangermeme.seqlet import tfmodisco_seqlets
from tangermeme.seqlet import recursive_seqlets

In [ ]:
#take all atac sequnce an put into tf mdoisco format
att, names = [], []
for name, ism_res in ism_enhancer.items():
    att.append(np.array(ism_res["ATAC"].sum(axis = 1)))
    names.append(name) 

atac_att = torch.tensor(np.stack(att))
atac_att

In [ ]:
#make one how encoding
atac_stacked = torch.tensor(np.stack([ism_res["ATAC"] for name, ism_res in ism_enhancer.items()]))
indices = torch.argmax(torch.abs(atac_stacked), dim=-1)
onehot = torch.nn.functional.one_hot(indices, num_classes=4).permute(0, 2, 1)
onehot.shape

In [ ]:
#run tfmodisco
#seqlets = tfmodisco_seqlets(atac_att, window_size = 15, flank=8, target_fdr=0.3, max_passing_frac=0.8)
seqlets = recursive_seqlets(atac_att, min_seqlet_len=6, max_seqlet_len=12, threshold=0.05)

In [ ]:
from tangermeme.io import read_meme
from tangermeme.annotate import annotate_seqlets

meme = read_meme(paths.META / "JASPAR2026_CORE_vertebrates_non-redundant_pfms_meme.txt")
meme_names = list(meme.keys())

m_idx, m_p = annotate_seqlets(onehot, seqlets, meme, n_nearest=10)

In [ ]:
#meme = read_meme(paths.META / "HOCOMOCO_H14_CORE.meme.txt")
#meme = read_meme(paths.META / "JASPAR2026_Baumann.meme.txt")

#meme = read_meme(paths.META / "TF-MODISCO.meme.txt")


In [ ]:
collapsed_p = pd.Series([tensor.item() for sublist in m_p for tensor in sublist])
plt.plot(-np.log10(collapsed_p.sort_values(ascending=False).values))
plt.xlabel("Hit (Sorted by pval)")
plt.ylabel("-log10(pval)")
#plt.hlines(y=-10, xmin = 0, xmax=len(seqlets))
plt.hlines(y=-np.log10(0.001), xmin = 0, xmax=len(collapsed_p))
plt.title("Pval per Hit")
plt.show()

In [ ]:
enhancer_motif_idx, enhancer_motif_pval = [], []
for motif_idxs, pvals in zip(m_idx, m_p):
    meme_idx, meme_pvals = [], []
    for mi, mp in zip(motif_idxs, pvals):
        meme_idx.append(mi)
        meme_pvals.append(mp)
    enhancer_motif_idx.append(meme_idx)
    enhancer_motif_pval.append(meme_pvals)

In [ ]:
len(enhancer_motif_idx)

In [ ]:
motif_names, m_p = [], []
for mi, mp in zip(enhancer_motif_idx, enhancer_motif_pval):
    meme_idx, meme_pvals = [], []
    if len(mi) != 0:
        for i, p in zip(mi, mp):
            if p.item() <= 0.001:
                name = meme_names[i.item()].upper().split(" ")[1]
                meme_idx.append(name)
                meme_pvals.append(p.item())
        motif_names.append(meme_idx), m_p.append(meme_pvals)
    else:
        motif_names.append(meme_idx), m_p.append(meme_pvals)
        

In [ ]:
seqlets["seq_idx"] = seqlets["example_idx"]
seqlets["example_idx"] = [str([name for name in names]) for names in motif_names]
seqlets["annot_p_val"] =  [str([p for p in ps]) for ps in m_p]
seqlets

In [ ]:
def trim_seqlets(seqlets, attributions, cutoff=0.8):
    '''Function to trim seqlet edges so that a base explains at least the cutoff percentage of the average attribution for that base'''
    rows = []
    for seq in seqlets.itertuples(index=False):
        s = seq.start
        e = seq.end
        l = e - s
        seq_id = seq.seq_idx
        sum_a = seq.attribution
        avg_a_per_pos = sum_a / l
        
        seq_attributions = attributions[seq_id, s:e]
        base_contributing = torch.where(seq_attributions / avg_a_per_pos > cutoff)[0]        
        row_data = seq._asdict()
        
        # Safety check ensure there is at least six bases meet the cutoff
        if len(base_contributing) > 5:
            first_true = base_contributing[0].item()
            last_true = base_contributing[-1].item() + 1 
            new_s = s + first_true
            new_e = s + last_true
            row_data['start'] = new_s
            row_data['end'] = new_e
               
            row_data['attribution'] = attributions[seq_id, new_s:new_e].sum().item()           
            rows.append(row_data)
        else:
            rows.append(row_data) #pass the og seqlet
            
    trimmed_seqlets = pd.DataFrame(rows)    
    return trimmed_seqlets

import ast
import pandas as pd



In [ ]:
trimmed_seqlets = trim_seqlets(seqlets, atac_att, cutoff=0.7)

In [ ]:
def simplify_tf_name(raw_name: str, families: set) -> str:
    raw_name_lower = raw_name.upper()
    
    # Exact match
    if raw_name_lower in families:
        return raw_name_lower
    # Partial match: the longest listed name contained in the motif name, ties
    # going to the one written first (DDIT3::CEBPA -> DDIT3). Looping over the
    # set directly left ties to string hashing, which changes between sessions.
    hits = [family for family in families if family in raw_name_lower]
    if hits:
        return min(hits, key=lambda family: (-len(family), raw_name_lower.find(family)))
            
    # No match
    return raw_name

def tidy_tf_names(df, tf_names, tf_col="example_idx", pval_col="annot_p_val"):
    tf_names_lower = {name.upper() for name in tf_names}

    def parse(val):
        if isinstance(val, list): return val
        try: return ast.literal_eval(val)
        except: return []

    selected_tfs, selected_pvals = [], []

    for _, row in df.iterrows():
        tfs = parse(row[tf_col])
        pvals = parse(row[pval_col])
 
        if not tfs: #halde case were no motif was found in jaspar aka empty list
            selected_tfs.append("Not found")
            selected_pvals.append(float("nan"))
        elif len(tfs) == 0: #halde case were no motif was found in jaspar aka empty list
            selected_tfs.append("Not found")
            selected_pvals.append(float("nan"))
        else:
            #check for matches
            matches = [(i, simplify_tf_name(tf, tf_names_lower)) for i, tf in enumerate(tfs) if simplify_tf_name(tf, tf_names_lower) in tf_names_lower]
            if len(matches) == 0:
                selected_tfs.append(tfs[0])
                selected_pvals.append(pvals[0])
            else:
                selected_tfs.append(matches[0][1])
                selected_pvals.append(pvals[0])

    return (
        pd.Series(selected_tfs, index=df.index, name="tf_name"),
        pd.Series(selected_pvals, index=df.index, name="pval"),
    )

In [ ]:
trimmed_seqlets["example_idx_alterntive"], trimmed_seqlets["annot_p_val_alterntive"] = trimmed_seqlets["example_idx"].copy(), trimmed_seqlets["annot_p_val"].copy()
trimmed_seqlets["example_idx"], trimmed_seqlets["annot_p_val"] = tidy_tf_names(trimmed_seqlets, all_TFs)


In [ ]:
trimmed_seqlets = trimmed_seqlets[trimmed_seqlets.example_idx != "Not found"]
trimmed_seqlets

In [ ]:
plt.plot(trimmed_seqlets.attribution.sort_values(ascending=True).values)
plt.xlabel("Hit (Sorted by Attribution)")
plt.ylabel("Attribution")
#plt.hlines(y=-10, xmin = 0, xmax=len(seqlets))
plt.hlines(y=3, xmin = 0, xmax=len(trimmed_seqlets))
plt.title("Sum Attribution per Hit")
plt.show()

In [ ]:
df = trimmed_seqlets
max_co = 10
cutoffs = np.arange(0, max_co, 0.25)
 
counts = {1: [], 2: [], 3: [], "4+": []}
 
for cutoff in cutoffs:
    passing = df[df["attribution"] > cutoff]
    per_seq = passing.groupby("seq_idx").size()
 
    counts[1].append((per_seq == 1).sum())
    counts[2].append((per_seq == 2).sum())
    counts[3].append((per_seq == 3).sum())
    counts["4+"].append((per_seq >= 4).sum())
 
fig, ax = plt.subplots(figsize=(12, 5))
 
colors = {"1 seqlet": "#185FA5", "2 seqlets": "#1D9E75", "3 seqlets": "#BA7517", "4+ seqlets": "black"}
labels = ["1 seqlet", "2 seqlets", "3 seqlets", "4+ seqlets"]
data_series = [counts[1], counts[2], counts[3], counts["4+"]]
 
bottom = np.zeros(len(cutoffs))
for label, series, color in zip(labels, data_series, colors.values()):
    bars = np.array(series)
    ax.bar(cutoffs, bars, bottom=bottom, width=0.25, label=label, color=color)
    bottom += bars
 
ax.set_xlabel("Attribution cutoff", fontsize=12)
ax.set_ylabel("Number of sequences with Seqlet", fontsize=12)
ax.set_title("Sequences with seqlets above attribution cutoff", fontsize=13)
ax.set_xticks(np.arange(0, max_co+1, 0.5))
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
 
plt.tight_layout()
plt.savefig(paths.FIGURES / "seqlet_stacked_bar.png", dpi=150)
plt.show()

In [ ]:
trimmed_seqlets = trimmed_seqlets[trimmed_seqlets.attribution >= 2]

In [ ]:
from tangermeme.plot import plot_logo

In [ ]:
seq_idx = 0
print(names[seq_idx])
ifg, ax = plt.subplots(1, 1, figsize=(20, 4))
ism_res = torch.tensor(ism_enhancer[names[seq_idx]]["ATAC"]).permute(1, 0)
plot_logo(ism_res, ax = ax,
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == seq_idx, ["example_idx_alterntive", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)

plt.tight_layout()
plt.show()    

In [ ]:
seq_idx = 0
print(names[seq_idx])
ifg, ax = plt.subplots(1, 1, figsize=(20, 4))
ism_res = torch.tensor(ism_enhancer[names[seq_idx]]["RNA ALDH1A1"]).permute(1, 0)
plot_logo(ism_res, ax = ax,
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == seq_idx, ["example_idx_alterntive", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)

plt.tight_layout()
plt.show()    

In [ ]:
seq_idx = 0
print(names[seq_idx])
ifg, ax = plt.subplots(1, 1, figsize=(20, 4))
ism_res = torch.tensor(ism_enhancer[names[seq_idx]]["TF STAT5A"]).permute(1, 0)
plot_logo(ism_res, ax = ax,
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == seq_idx, ["example_idx_alterntive", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)
plt.tight_layout()
plt.show()    

In [ ]:
seq_idx = 68
print(names[seq_idx])
ifg, ax = plt.subplots(1, 1, figsize=(20, 4))
ism_res = torch.tensor(ism_enhancer[names[seq_idx]]["ATAC"]).permute(1, 0)
plot_logo(ism_res, ax = ax,
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == seq_idx, ["example_idx_alterntive", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)
plt.tight_layout()
plt.show()    

In [ ]:
seq_idx = 68
print(names[seq_idx])
ifg, ax = plt.subplots(1, 1, figsize=(20, 4))
ism_res = torch.tensor(ism_enhancer[names[seq_idx]]["TF STAT5A"]).permute(1, 0)
plot_logo(ism_res, ax = ax,
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == seq_idx, ["example_idx_alterntive", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)
plt.tight_layout()
plt.show()    

In [ ]:
seq_idx = 50
print(names[seq_idx])
ifg, ax = plt.subplots(1, 1, figsize=(20, 4))
ism_res = torch.tensor(ism_enhancer[names[seq_idx]]["ATAC"]).permute(1, 0)
plot_logo(ism_res, ax = ax,
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == seq_idx, ["example_idx_alterntive", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)
plt.tight_layout()
plt.show()    

In [ ]:
ylim = atac_att.min().item(), atac_att.max().item()

df = trimmed_seqlets
multi_seq_idx = df.seq_idx.value_counts()[df.seq_idx.value_counts() > 1].index
df = df[df.seq_idx.isin(multi_seq_idx)]

fig, axes = plt.subplots(df.seq_idx.nunique(), 1, figsize = (20, 50))
print(len(axes))

for ax, seq_i in enumerate(np.sort(df.seq_idx.unique())):
    name = names[seq_i]
    ism_res = torch.tensor(ism_enhancer[name]["ATAC"]).permute(1, 0)
    plot_logo(ism_res, ax=axes[ax], annotations=df[df.seq_idx == seq_i], score_key='attribution', n_tracks=5)
    axes[ax].set_ylim(ylim)
    axes[ax].set_ylabel(f"{name} {seq_i}", rotation=0)

plt.tight_layout()
plt.show()    

# Seqlet scramble analysis

In [ ]:
from pyfaidx import Fasta

if paths.HG38_FASTA.exists():
    hg38 = Fasta(str(paths.HG38_FASTA))
    enhancers["seq"] = [str(hg38[e.chrom][e.start:e.end]) for e in enhancers.itertuples()]
else:
    # Without the reference the enhancer sequences are read from the ISM
    # tracks: attributions are projected onto the reference base, so the argmax
    # of |ISM| is the genomic sequence (verified on every enhancer).
    hg38 = None
    _nuc = np.array(list("ACGT"))
    enhancers["seq"] = [
        "".join(_nuc[tc.reference_onehot_from_ism(ism_enhancer[n]["ATAC"]).argmax(axis=0)])
        for n in enhancers["name"]]
    if RUN_PREDICTIONS:
        raise FileNotFoundError(f"RUN_PREDICTIONS needs {paths.HG38_FASTA}")


In [ ]:
from tangermeme.utils import one_hot_encode


"""
Generate all seqlet scramble combinations per enhancer.

For each enhancer with n seqlets, produces 1 + (2^n - 1) * 10 sequences:
  - 1  row : all seqlets unscrambled  → original enhancer sequence
  - 10 rows: per non-trivial combo    → each of the 10 dinucleotide shuffles,
               with the scrambled-seqlet windows replaced from that shuffle

The `dishuff` tensor must already be loaded in scope:
    dishuff.shape == [95, 10, 4, 313]  (n_enhancers, n_shuffles, 4, seq_len)
"""

import itertools
import numpy as np
import pandas as pd
import torch

# ACGT order assumed (standard one-hot for genomics)
_IDX_TO_NUC = np.array(["A", "C", "G", "T"])


def onehot_to_seq(onehot: torch.Tensor) -> str:
    """
    Convert a one-hot tensor of shape [4, L] to an uppercase nucleotide string.
    Handles ties by taking argmax (first max index).
    """
    indices = onehot.argmax(dim=0).numpy()   # shape [L]
    return "".join(_IDX_TO_NUC[indices])


def splice_scramble(
    original_seq: str,
    scramble_windows: list[tuple[int, int]],
    donor_seq: str,
) -> str:
    """
    Replace every (start, end) window in `original_seq` with the same window
    from `donor_seq`.  start/end are 0-based, end exclusive (Python slicing).
    """
    seq = list(original_seq)
    for start, end in scramble_windows:
        seq[start:end] = list(donor_seq[start:end])
    return "".join(seq)


def generate_combination_df(
    motif_df: pd.DataFrame,
    dishuff: torch.Tensor,
    enhancer_col: str = "name",
    seq_col: str = "seq",
    seq_idx_col: str = "seq_idx",
    start_col: str = "start_seq",
    end_col: str = "end_seq",
    design: str = "reduced",
    seqlet_name_col: str = "example_idx",   # human-readable seqlet label
) -> pd.DataFrame:
    """
    Parameters
    ----------
    motif_df    : DataFrame as described (one row per seqlet)
    dishuff     : Tensor [n_enhancers, 10, 4, seq_len], dinucleotide shuffles
    *_col       : column name overrides

    Returns
    -------
    pd.DataFrame with columns:
        name, seq_idx, combo_id, combo_label,
        scrambled_seqlets, unscrambled_seqlets,
        n_scrambled, shuffle_idx, sequence
    """
    n_shuffles = dishuff.shape[1]
    rows = []

    for enhancer, grp in motif_df.groupby(enhancer_col):
        grp = grp.reset_index(drop=True)

        original_seq = grp[seq_col].iloc[0].upper()
        seq_idx      = int(grp[seq_idx_col].iloc[0])

        # Seqlets for this enhancer, sorted by start position
        grp_sorted = grp.sort_values(start_col).reset_index(drop=True)
        n_seqlets  = len(grp_sorted)

        # Pre-convert all 10 shuffled tensors to strings once
        enhancer_shuffles = dishuff[seq_idx]          # [10, 4, L]
        shuffle_seqs = [
            onehot_to_seq(enhancer_shuffles[i])
            for i in range(n_shuffles)
        ]

        # Seqlet metadata lists (aligned with grp_sorted rows)
        seqlet_labels  = grp_sorted[seqlet_name_col].tolist()
        seqlet_windows = list(
            zip(grp_sorted[start_col].tolist(), grp_sorted[end_col].tolist())
        )

        # Iterate over all 2^n bitmask combinations
        # bit i == 1  →  seqlet i is SCRAMBLED
        
        for combo_id, combo in enumerate(tc.enumerate_combinations(n_seqlets, design)):
            scrambled_idx   = list(combo)
            unscrambled_idx = [i for i in range(n_seqlets) if i not in combo]
        
#         for combo_id in range(2 ** n_seqlets):
#             bits = [(combo_id >> i) & 1 for i in range(n_seqlets)]

#             scrambled_idx   = [i for i, b in enumerate(bits) if b]
#             unscrambled_idx = [i for i, b in enumerate(bits) if not b]

            scrambled_labels   = [seqlet_labels[i]  for i in scrambled_idx]
            unscrambled_labels = [seqlet_labels[i]  for i in unscrambled_idx]
            scrambled_windows  = [seqlet_windows[i] for i in scrambled_idx]

            # Human-readable label, e.g. "scr[GATA2,STAT5A]_unscr[MYB73]"
            combo_label = (
                f"scr[{','.join(str(s) for s in scrambled_labels)}]"
                f"_unscr[{','.join(str(s) for s in unscrambled_labels)}]"
            )

            if combo_id == 0:
                # ── all unscrambled: use original sequence, no shuffle ──
                rows.append(
                    dict(
                        name       = enhancer,
                        seq_idx             = seq_idx,
                        combo_id            = combo_id,
                        combo_label         = combo_label,
                        scrambled_seqlets   = scrambled_labels,
                        unscrambled_seqlets = unscrambled_labels,
                        n_scrambled         = 0,
                        shuffle_idx         = None,
                        sequence            = original_seq,
                    )
                )
            else:
                # ── scrambled combo: one row per shuffle ──
                for shuf_i, donor in enumerate(shuffle_seqs):
                    scrambled_seq = splice_scramble(
                        original_seq, scrambled_windows, donor
                    )
                    rows.append(
                        dict(
                            name       = enhancer,
                            seq_idx             = seq_idx,
                            combo_id            = combo_id,
                            combo_label         = combo_label,
                            scrambled_seqlets   = scrambled_labels,
                            unscrambled_seqlets = unscrambled_labels,
                            n_scrambled         = len(scrambled_idx),
                            shuffle_idx         = shuf_i,
                            sequence            = scrambled_seq,
                        )
                    )

    result = pd.DataFrame(rows)

    # Tidy column order
    col_order = [
        "name", "seq_idx", "combo_id", "combo_label",
        "n_scrambled", "scrambled_seqlets", "unscrambled_seqlets",
        "shuffle_idx", "sequence",
    ]
    return result[col_order]

"""
Pad generated enhancer sequences to a full model-input length.

Strategy
--------
For each row in `combo_df`:
  1. Compute the genomic center of the enhancer  (chrom, start, end)
  2. Fetch a window of `input_length` bp from hg38 centered on that point
  3. Splice the generated sequence into the center of that window,
     replacing exactly the enhancer region
  4. Return an uppercase string of length `input_length`

Handles near-chromosome-boundary edge cases by N-padding.

Dependencies
------------
    pyfaidx  (pip install pyfaidx)
    pandas, numpy
"""

import numpy as np
import pandas as pd
from pyfaidx import Fasta


def make_model_inputs(
    combo_df: pd.DataFrame,
    hg38: Fasta,
    input_length: int = 2 ** 20,
    chrom_col: str = "chrom",
    start_col: str = "start",
    end_col: str = "end",
    seq_col: str = "sequence",
) -> list[str]:
    """
    Parameters
    ----------
    combo_df      : output of generate_combination_df(), one row per sequence
    hg38          : pyfaidx.Fasta handle to the hg38 genome
    input_length  : total length of the model input (e.g. 2**19 = 524_288)
    *_col         : column name overrides

    Returns
    -------
    List of uppercase strings of length `input_length`, in the same row order
    as combo_df.
    """
    results: list[str] = []

    for _, row in combo_df.iterrows():
        chrom     = row[chrom_col]
        enh_start = int(row[start_col])   # 0-based, BED-style
        enh_end   = int(row[end_col])     # 0-based exclusive
        enh_seq   = row[seq_col].upper()  # our generated sequence

        enh_len   = enh_end - enh_start
        assert len(enh_seq) == enh_len, (
            f"Generated sequence length {len(enh_seq)} != "
            f"enhancer span {enh_len} for {chrom}:{enh_start}-{enh_end}"
        )

        # ── center the enhancer in the input window ──────────────────────────
        enh_center = (enh_start + enh_end) // 2

        win_start  = enh_center - input_length // 2
        win_end    = win_start  + input_length          # always exactly input_length

        # ── fetch genomic flanks, N-padding at chromosome boundaries ─────────
        chrom_len  = len(hg38[chrom])

        # How much to clip on each side
        left_pad  = max(0, -win_start)          # bp to N-pad on the left
        right_pad = max(0, win_end - chrom_len) # bp to N-pad on the right

        fetch_start = max(0,          win_start)
        fetch_end   = min(chrom_len,  win_end)

        genomic = hg38[chrom][fetch_start:fetch_end].seq.upper()

        # Rebuild with N-pads if near chromosome boundary
        if left_pad or right_pad:
            genomic = "N" * left_pad + genomic + "N" * right_pad

        assert len(genomic) == input_length, (
            f"Fetched {len(genomic)} bp, expected {input_length} "
            f"(chrom={chrom}, win={win_start}-{win_end})"
        )

        # ── splice generated sequence into window ─────────────────────────────
        # Position of enhancer relative to window start
        enh_in_win_start = enh_start - win_start + left_pad
        enh_in_win_end   = enh_in_win_start + enh_len

        full_seq = (
            genomic[:enh_in_win_start]
            + enh_seq
            + genomic[enh_in_win_end:]
        )

        assert len(full_seq) == input_length, (
            f"Output length {len(full_seq)} != {input_length}"
        )

        results.append(full_seq)

    return results

In [ ]:
#combine seqlets with enhacer data and onehot encode
trimmed_seqlets["name"] = trimmed_seqlets.seq_idx.map(dict(zip(range(0, len(names), 1) ,names)))
fulls_seqlets = pd.merge(trimmed_seqlets, enhancers, how = "left", on = "name", suffixes=("_seq", ""))

seq_1hot = torch.stack([one_hot_encode(seq) for seq in enhancers.seq])
seq_1hot.shape

In [ ]:
fulls_seqlets = tc.make_seqlet_labels_unique(
    fulls_seqlets, enhancer_col="name", label_col="example_idx", start_col="start_seq"
)
fulls_seqlets = tc.cap_seqlets_per_enhancer(
    fulls_seqlets, max_seqlets=5, enhancer_col="name",
    score_col="attribution", start_col="start_seq", end_col="end_seq",
)

budget = tc.summarise_design_budget(fulls_seqlets, n_shuffles=10, design="reduced")
print(budget.tail(1))

In [ ]:
#make raandomized shuffeled seqs
from tangermeme.ersatz import dinucleotide_shuffle

dishuff = dinucleotide_shuffle(seq_1hot, n=10, random_state=42)
dishuff.shape

In [ ]:
control_windows = tc.sample_control_windows(
    exclude_seqlets=fulls_seqlets,  
    attributions=atac_att,
    enhancer_names=names,
    n_controls=3,
    attribution_percentile=50.0,
)
control_windows = control_windows.merge(
    enhancers[["name", "chrom", "start", "end", "seq", "target_gene"]],
    on="name", how="left",
)
print(tc.summarise_design_budget(control_windows, design="singles").tail(1))

control_df = generate_combination_df(control_windows, dishuff, design="singles")
control_df = pd.merge(control_df, enhancers[["chrom","start","end","name","target_gene"]],
                      on="name", how="left")

In [ ]:
#genrate all pairs of seqlet disrubtion per enhancer with 10 random sequnces for scarmableing
out_df = generate_combination_df(fulls_seqlets, dishuff)

print(out_df.head(2).to_string())
print(f"\nTotal rows: {len(out_df)}")
print(f"Unique enhancers: {out_df['name'].nunique()}")

# Quick sanity check: all sequences same length per enhancer
seq_lens = out_df.groupby("name")["sequence"].apply(
    lambda s: s.str.len().unique()
)
assert all(len(v) == 1 for v in seq_lens), "Sequence length mismatch!"
print("All sequences per enhancer have consistent length.")


In [ ]:
out_df = pd.merge(out_df, enhancers[["chrom","start","end","name","target_gene"]], how = "left", on = "name")

In [ ]:
SEQ_LEN = dna_client.SEQUENCE_LENGTH_1MB

In [ ]:
from alphagenome_research.model.variant_scoring.gene_mask_extractor import (
    GeneMaskExtractor, GeneMaskType, GeneQueryType
)

# GENCODE v46, the annotation AlphaGenome uses; cached in data/reference.
gtf_df = paths.load_gtf()

gtf_df = gtf_df[gtf_df.gene_type == "protein_coding"]

exon_extractor = GeneMaskExtractor(
    gtf=gtf_df,
    gene_mask_type=GeneMaskType.EXONS,
    gene_query_type=GeneQueryType.INTERVAL_CONTAINED,
    filter_protein_coding=False
)

In [ ]:
#predict the effect on enahcer accessibilty and target genen expression
def mean_of_lists(series):
    """Element-wise mean across a series of lists."""
    parsed = [
        ast.literal_eval(v) if isinstance(v, str) else v
        for v in series
    ]
    return np.array(parsed).mean(axis=0).tolist()

def predict_seqlet_KO_effects(combinations_df, genome = hg38, seq_len = SEQ_LEN, model = model, enhancers = enhancers, curies = K562_curie):
    df_list = []
    df_list_raw = []
    number_enhs = len(combinations_df.name.unique())
    for enh_nr, name in enumerate(combinations_df.name.unique()):
        print(f"Analysing enhancer: {name} ({enh_nr}/{number_enhs})")
        enh_scram = combinations_df[combinations_df.name == name]

        model_inputs = make_model_inputs(enh_scram, genome, input_length=seq_len)

        # Sanity: verify the enhancer region is our sequence, not raw genome
        row0       = enh_scram.iloc[0]
        enh_center = (row0["start"] + row0["end"]) // 2
        win_start  = enh_center - seq_len // 2
        offset     = row0["start"] - win_start
        embedded   = model_inputs[0][offset : offset + (row0["end"] - row0["start"])]
        assert embedded == row0["sequence"].upper(), "Sanity check failed!"
        print("Sanity check passed.")

        enhancer_length =  (enhancers[enhancers.name == name].end.values - enhancers[enhancers.name == name].start.values)[0]
        input_interval = enhancers[enhancers.name == name].ivs.values[0].resize(seq_len)


        out_pred = tc.predict_with_backoff(
            model=model,
            sequences=model_inputs,
            interval=input_interval,
            requested_outputs=[dna_client.OutputType["ATAC"], dna_client.OutputType["RNA_SEQ"]],
            ontology_terms=curies,
            chunk_size=64,
            max_workers=6,
        )

        #get gene masks once for enhancer
        gene_mask, annot = exon_extractor.extract(interval = out_pred[0].rna_seq.interval)
        is_target = annot.gene_name.isin(enhancers[enhancers.name == name].target_gene.values[0])
        annot = annot.loc[is_target]
        gene_mask = gene_mask[:, is_target]

        #ATAC aggreagarion mask
        agg_start = (seq_len - enhancer_length) // 2
        agg_end = agg_start + enhancer_length


        #iterate ove rshuffels
        target_genes, RNA_counts, ATAC_counts = [], [], []
        for seq_out in out_pred:

            #summ aatac coutns
            ATAC_counts.append(seq_out.atac.values.mean(axis = 1)[agg_start:agg_end].sum())

            #sum gene counts       
            targets, counts = [], []
            for mask_idx, gene in enumerate(annot.itertuples()):
                strand = gene.strand
                stranded_track = seq_out.rna_seq.metadata.strand.isin([strand, "."])
                gene_counts = seq_out.rna_seq.values[:,stranded_track].mean(axis = 1)[gene_mask[:, mask_idx]].mean()

                targets.append(gene.gene_name)
                counts.append(gene_counts)
            target_genes.append(targets)
            RNA_counts.append(counts)

        del out_pred
        enh_scram["ATAC_counts"], enh_scram["RNA_counts"], enh_scram["target_gene"], =  ATAC_counts, RNA_counts, target_genes

        #aggregate the shuffeld ref sequnces by avg
        df_list_raw.append(enh_scram.copy())
        enh_scram_agg = (
            enh_scram
            .groupby(
                ["name", "seq_idx", "combo_label", "n_scrambled", "chrom", "start", "end"],
                observed=True,
            )
            .agg(
                ATAC_counts         = ("ATAC_counts",         "mean"),
                RNA_counts          = ("RNA_counts",           mean_of_lists),
                target_gene         = ("target_gene",          "first"),
                scrambled_seqlets   = ("scrambled_seqlets",    "first"),
                unscrambled_seqlets = ("unscrambled_seqlets",  "first"),
            )
            .reset_index()
        )

        df_list.append(enh_scram_agg)
        del enh_scram, enh_scram_agg
        
    return df_list_raw, df_list

In [ ]:
if RUN_PREDICTIONS:
    control_list_raw, control_list = predict_seqlet_KO_effects(control_df)


In [ ]:
if RUN_PREDICTIONS:
    ctrl_effects_raw = pd.concat(control_list_raw).reset_index(drop=True)
    ctrl_effects     = pd.concat(control_list).reset_index(drop=True)
    ctrl_effects_raw.to_parquet(
        paths.PROCESSED / "TF_insilico_KO_controls_raw.parquet",
        engine='pyarrow', index=False)
    ctrl_effects.to_parquet(
        paths.PROCESSED / "TF_insilico_KO_controls.parquet",
        engine='pyarrow', index=False)


In [ ]:
if RUN_PREDICTIONS:
    TF_seqlet_list_raw, TF_seqlet_list = predict_seqlet_KO_effects(out_df)


In [ ]:
if RUN_PREDICTIONS:
    scram_effects_raw = pd.concat(TF_seqlet_list_raw).reset_index(drop=True)
    scram_effects     = pd.concat(TF_seqlet_list).reset_index(drop=True)
    scram_effects_raw.to_parquet(
        paths.PROCESSED / "TF_insilico_KO_raw.parquet",
        engine='pyarrow', index=False)
    scram_effects.to_parquet(
        paths.PROCESSED / "TF_insilico_KO.parquet",
        engine='pyarrow', index=False)


In [ ]:
#load the predictins
#scram_effects = pd.read_parquet(paths.PROCESSED / "TF_insilico_KO.parquet", engine='pyarrow')

# Plot effects

In [ ]:
scram_effects_raw = pd.read_parquet(
    paths.PROCESSED / "TF_insilico_KO_raw.parquet",
    engine='pyarrow')

ctrl_effects_raw = pd.read_parquet(
    paths.PROCESSED / "TF_insilico_KO_controls_raw.parquet",
    engine='pyarrow')

# The seqlet and control tables are recomputed from the ISM file on every run,
# so check they reproduce the labels and indices the stored predictions used. A
# different tangermeme version can shift seqlet boundaries and break this.
_stored = {lab for labs in scram_effects_raw["scrambled_seqlets"] for lab in labs}
_missing = _stored - set(fulls_seqlets["example_idx"])
assert not _missing, f"stored seqlet labels not reproduced: {sorted(_missing)[:10]}"
_idx = scram_effects_raw.drop_duplicates("name").set_index("name")["seq_idx"]
assert all(names[i] == n for n, i in _idx.items()), "seq_idx order differs from the ISM file"


In [ ]:
#analyse
motifs    = tc.add_bh_correction(tc.build_motif_effects(scram_effects_raw, "ATAC_counts", seqlets=fulls_seqlets))
pairs     = tc.add_bh_correction(tc.build_pairwise_interactions(scram_effects_raw, "ATAC_counts", seqlets=fulls_seqlets))
coop_atac = tc.add_bh_correction(tc.build_cooperativity_table(scram_effects_raw, "ATAC_counts"))
coop_rna  = tc.add_bh_correction(tc.build_cooperativity_table(scram_effects_raw, "RNA_counts"))

controls  = tc.build_motif_effects(ctrl_effects_raw, "ATAC_counts", seqlets=control_windows)

In [ ]:
importlib.reload(tp)


In [ ]:
fig = tp.make_main_figure(coop_atac, coop_rna, pairs, motifs, controls=controls, figsize=(10, 12))
#fig.savefig("figures/tf_cooperativity_main.pdf", bbox_inches="tight")
plt.show()

In [ ]:
split    = tc.build_saturation_split(scram_effects_raw, value_col="ATAC_counts")
floor_df = tc.build_floor_check(scram_effects_raw, value_col="ATAC_counts")
dose     = tc.build_dose_response(scram_effects_raw, value_col="ATAC_counts", seqlets=fulls_seqlets)

In [ ]:
fig = tp.make_supplementary_figure(motifs, coop_atac, pairs, controls=controls,
                                   split=split, floor_df=floor_df, dose=dose,
                                  figsize = (10, 14))
plt.show()

In [ ]:
#partial orrealtion regress out effect siezcodniiton for motif spacing
spacing = cl.summarise_spacing_partial(
    pairs,                    # tc.build_pairwise_interactions(..., seqlets=...)
    spacing_col="spacing_bp", value_col="interaction",
    delta_a_col="delta_a", delta_b_col="delta_b",
)

In [ ]:
spacing

# Save analysis

In [ ]:
from element_epistasis.pipeline import run_all
tables = run_all(
    tf_scram=scram_effects_raw,
    tf_seqlets=fulls_seqlets,
    tf_control_scram=ctrl_effects_raw,
    tf_control_windows=control_windows,
    top_n=20, 
    summary_path=paths.SUMMARIES / "within_enhancer_TF_summary.txt",
)

# Run the Cross Enhacer TF TF interaction

In [ ]:
SEQ_LEN = dna_client.SEQUENCE_LENGTH_1MB   # 1,048,576 bp
ENH_LEN = 313                              # every enhancer in the bed is 313 bp

MAX_SEQLETS_PER_ENHANCER = 5               # your decision
MAX_SEQLETS_PER_GENE = 20                  # your decision
N_SHUFFLES = 10                            # matches the within-enhancer run
TOP_K_MATCHES = 5                          #number of motif pere seqlts to match TFs with

# Include within-enhancer pairs alongside cross-enhancer pairs, so that the two
# are measured in the SAME 1 Mb window and are therefore directly comparable.
# Comparing against the previous run instead would confound the biology with
# the change of window anchoring.
INCLUDE_WITHIN_ENHANCER_PAIRS = True

# Rows of the combination table converted to 1 Mb strings at once. Each string
# is ~1 MB, so this bounds peak memory at roughly INPUT_BLOCK megabytes.
INPUT_BLOCK = 128

In [ ]:
#load the singel enahcer predcited KO effect on RNA fromthe enahcer enahcer GLM resutls
enh_mains = pd.read_csv(paths.PROCESSED / "enhancer_insilico_single_KO_main_effects.csv")

In [ ]:
# ASSUMPTION: dishuff is indexed by the same seq_idx used in fulls_seqlets, and
# holds N_SHUFFLES dinucleotide shuffles of each 313 bp enhancer.
assert dishuff.shape[1] >= N_SHUFFLES, "dishuff has fewer shuffles than requested"
assert dishuff.shape[-1] == ENH_LEN, f"dishuff sequence length is not {ENH_LEN}"

In [ ]:
# ---------------------------------------------------------------------------
# 1. Gene-level enhancer table
#
# A bed name such as "ALDH1A1_enh2:ANXA1_enh1" denotes ONE genomic region that
# serves as enh2 of ALDH1A1 and enh1 of ANXA1. Exploding on the colon therefore
# produces two rows that share coordinates, a sequence and a seq_idx, but carry
# different gene-qualified labels. That region will be perturbed independently
# in each gene's design, which is correct: the two genes have different windows.
# ---------------------------------------------------------------------------

# seq_idx must come from the ISM key order, because that is what indexes dishuff.
name_to_seq_idx = {name: i for i, name in enumerate(names)}

gene_enh_rows = []
for _, row in enhancers.iterrows():
    # Each part of the compound name is one "GENE_enhN" label.
    for part in str(row["name"]).split(":"):
        gene = part.split("_")[0]
        gene_enh_rows.append(dict(
            gene=gene,
            enh_key=part,                 # gene-qualified, unique project-wide
            bed_name=row["name"],         # the ISM key, may be compound
            seq_idx=name_to_seq_idx.get(row["name"], np.nan),
            chrom=row["chrom"],
            start=int(row["start"]),
            end=int(row["end"]),
            seq=row["seq"].upper(),
        ))

gene_enhancers = pd.DataFrame(gene_enh_rows)

# A missing seq_idx means the enhancer has no ISM and therefore no seqlets.
missing = gene_enhancers["seq_idx"].isna()
if missing.any():
    print(f"[warn] {int(missing.sum())} enhancer(s) absent from the ISM set, dropped:")
    print(gene_enhancers.loc[missing, 'enh_key'].tolist())
    gene_enhancers = gene_enhancers[~missing].copy()
gene_enhancers["seq_idx"] = gene_enhancers["seq_idx"].astype(int)

print(f"genes={gene_enhancers.gene.nunique()}  "
      f"gene-enhancer pairs={len(gene_enhancers)}")
print(gene_enhancers.groupby("gene").size().describe()[["min", "50%", "max"]])

In [ ]:
gene_enhancers

In [ ]:
# ---------------------------------------------------------------------------
# 2. Gene windows
#
# The window must contain the gene body AND every enhancer assigned to it, so
# that one forward pass sees all perturbations and the target gene together.
# Centre on the midpoint of that union, mirroring the enhancer-level KO design.
# ---------------------------------------------------------------------------

# ASSUMPTION: gtf_df carries gene_name, seqname/chromosome, start and end.
# Column naming differs between GENCODE feather releases, so resolve it once.
_chrom_col = next(c for c in ["Chromosome", "seqname", "chrom", "contig"]
                  if c in gtf_df.columns)
gene_bodies = (gtf_df[gtf_df["gene_name"].isin(gene_enhancers["gene"].unique())]
               .groupby("gene_name")
               .agg(gene_chrom=(_chrom_col, "first"),
                    gene_start=("Start" if "Start" in gtf_df.columns else "start", "min"),
                    gene_end=("End" if "End" in gtf_df.columns else "end", "max")))

window_rows = []
for gene, group in gene_enhancers.groupby("gene"):
    chrom = group["chrom"].iloc[0]
    lo, hi = int(group["start"].min()), int(group["end"].max())

    # Extend to cover the gene body when the annotation is available. If the
    # gene is not in the GTF the enhancer span alone is used, which still
    # centres the window sensibly but is worth knowing about.
    if gene in gene_bodies.index:
        body = gene_bodies.loc[gene]
        if str(body["gene_chrom"]).replace("chr", "") == str(chrom).replace("chr", ""):
            lo = min(lo, int(body["gene_start"]))
            hi = max(hi, int(body["gene_end"]))
    else:
        print(f"[warn] {gene} not found in GTF; window built from enhancers only")

    centre = (lo + hi) // 2
    win_start = centre - SEQ_LEN // 2
    window_rows.append(dict(gene=gene, chrom=chrom, span_bp=hi - lo,
                            win_start=win_start, win_end=win_start + SEQ_LEN,
                            n_enhancers=len(group)))

gene_windows = pd.DataFrame(window_rows).set_index("gene")

# Everything must fit with margin; a truncated enhancer would silently corrupt
# the design rather than raise.
too_wide = gene_windows[gene_windows["span_bp"] > SEQ_LEN - 2 * ENH_LEN]
assert too_wide.empty, f"gene span exceeds the window:\n{too_wide}"

# Genes with one enhancer cannot contribute a cross-enhancer pair.
single = gene_windows[gene_windows["n_enhancers"] < 2]
if not single.empty:
    print(f"[info] dropping {len(single)} single-enhancer gene(s): "
          f"{single.index.tolist()}")
    gene_windows = gene_windows.drop(single.index)
    gene_enhancers = gene_enhancers[gene_enhancers.gene.isin(gene_windows.index)]

print(f"genes retained: {len(gene_windows)}")
print(f"span (kb): median={gene_windows.span_bp.median()/1000:.0f}  "
      f"max={gene_windows.span_bp.max()/1000:.0f}")


In [ ]:
# ---------------------------------------------------------------------------
# 3. TF eligibility from the top K motif matches
#
# here we restric the naming to the top k alterantive seqltes annoations


# Canonicalise the cooperativity list: tfs.tsv contains duplicates and
# non-standard spellings, so compare in upper case after stripping whitespace.
coop_tf_set = {str(t).strip().upper() for t in all_TFs if str(t).strip()}
print(f"cooperativity TF list: {len(coop_tf_set)} unique symbols")

def _eligible_from_matches(match_list) -> bool:
    """True if any of the top K matches simplifies to a listed factor."""
    if not isinstance(match_list, (list, tuple)):
        return False
    for raw in list(match_list)[:TOP_K_MATCHES]:
        if simplify_tf_name(str(raw), coop_tf_set) in coop_tf_set:
            return True
    return False

seqlets_annot = fulls_seqlets.copy()

seqlets_annot["example_idx_alterntive"] = seqlets_annot[
    "example_idx_alterntive"
].apply(ast.literal_eval)

if "example_idx_alterntive" in seqlets_annot.columns:
    seqlets_annot["top_matches"] = seqlets_annot["example_idx_alterntive"].apply(
        lambda v: list(v)[:TOP_K_MATCHES] if isinstance(v, (list, tuple)) else [])
    seqlets_annot["tf_eligible"] = seqlets_annot["top_matches"].apply(
        _eligible_from_matches)
    print("eligibility computed from the top three ranked matches")
else:
    print("[warn] `seqlets_annot[example_idx_alterntive]` unavailable; falling back to the primary label. "
          "Eligibility will be STRICTER than the top-three rule intends.")
    seqlets_annot["top_matches"] = seqlets_annot["example_idx"].apply(lambda v: [v])
    seqlets_annot["tf_eligible"] = seqlets_annot["example_idx"].apply(
        lambda v: simplify_tf_name(str(v), coop_tf_set) in coop_tf_set)

print(f"eligible seqlets: {int(seqlets_annot.tf_eligible.sum())}/"
      f"{len(seqlets_annot)} ({seqlets_annot.tf_eligible.mean():.1%})")

In [ ]:
coop_tf_set

In [ ]:
# ---------------------------------------------------------------------------
# 4. Seqlet coordinates in genomic space, and gene-qualified labels
#
# start_seq / end_seq are offsets INSIDE the 313 bp enhancer sequence. The
# gene-level window needs genomic coordinates. A silent off-by-one here would
# scramble the wrong bases and nothing downstream would reveal it, so both the
# containment and the round trip are asserted.
#
# Labels become "GENE_enhN|MOTIF" so that the same factor occurring in two
# enhancers of one gene cannot collapse into a single design cell, and so that
# a pair can be classified as within or across by parsing the prefix.
# ---------------------------------------------------------------------------

# Join seqlets onto the exploded gene table. A shared enhancer produces one
# seqlet row per gene, which is intended.
seqlets_gene = seqlets_annot.merge(
    gene_enhancers[["gene", "enh_key", "bed_name", "seq_idx", "chrom",
                    "start", "end", "seq"]],
    left_on="name", right_on="bed_name", how="inner", suffixes=("", "_enh"),
)

# Enhancer-local -> genomic.
seqlets_gene["genomic_start"] = seqlets_gene["start"] + seqlets_gene["start_seq"]
seqlets_gene["genomic_end"] = seqlets_gene["start"] + seqlets_gene["end_seq"]
seqlets_gene["label"] = (seqlets_gene["enh_key"] + "|"
                         + seqlets_gene["example_idx"].astype(str))

# Every seqlet must sit inside its parent enhancer.
assert (seqlets_gene["start_seq"] >= 0).all(), "negative seqlet start"
assert (seqlets_gene["end_seq"] <= ENH_LEN).all(), "seqlet runs past enhancer end"
assert (seqlets_gene["genomic_start"] >= seqlets_gene["start"]).all()
assert (seqlets_gene["genomic_end"] <= seqlets_gene["end"]).all()

# Labels must be unique within a gene, otherwise the design silently merges cells.
dupes = seqlets_gene.groupby(["gene", "label"]).size()
assert (dupes <= 1).all(), f"duplicate labels within a gene:\n{dupes[dupes > 1]}"

print(f"seqlets mapped: {len(seqlets_gene)} across "
      f"{seqlets_gene.gene.nunique()} genes")


In [ ]:
# ---------------------------------------------------------------------------
# 5. Seqlet selection
#
# Two-stage ranking, per your decision:
#   1. Enhancers are ranked within a gene by |beta_i|, the GLM main effect of
#      scrambling that whole enhancer on target gene expression. Motifs from
#      strong enhancers are taken first.
#   2. Within an enhancer, seqlets are ranked by ISM attribution, with
#      TF-list-eligible seqlets promoted above non-eligible ones.
#
# Caps: MAX_SEQLETS_PER_ENHANCER then MAX_SEQLETS_PER_GENE.
#
# NOTE: with at most 4 enhancers per gene and a per-enhancer cap of 4, the
# per-gene cap of 16 is only reached at the boundary, so in practice the
# enhancer ranking rarely causes a drop. The report below states how often it
# actually bound, so the effect of the rule is visible rather than assumed.
# ---------------------------------------------------------------------------

# ASSUMPTION: `enh_mains` is the enhancer-level GLM main effect table with
# `full_a` holding "GENE_enhN" and `coef` holding beta_i on RNA.
beta_lookup = (enh_mains.drop_duplicates("full_a")
                        .set_index("full_a")["coef"].abs())

selection = seqlets_gene.copy()
selection["enh_beta_abs"] = selection["enh_key"].map(beta_lookup)

# Enhancers absent from the GLM table get rank last rather than being dropped.
n_missing_beta = int(selection["enh_beta_abs"].isna().sum())
if n_missing_beta:
    print(f"[warn] {n_missing_beta} seqlet(s) on enhancers with no beta_i; "
          f"those enhancers are ranked last")
selection["enh_beta_abs"] = selection["enh_beta_abs"].fillna(-np.inf)

kept = []
for gene, group in selection.groupby("gene"):
    # Enhancers strongest first.
    enh_order = (group.groupby("enh_key")["enh_beta_abs"].first()
                      .sort_values(ascending=False).index.tolist())

    chosen_for_gene = []
    for enh_key in enh_order:
        block = group[group["enh_key"] == enh_key]
        # Eligible first, then by attribution. Non-overlap is enforced greedily,
        # since overlapping windows cannot be perturbed independently.
        block = block.sort_values(["tf_eligible", "attribution"],
                                  ascending=[False, False])
        accepted = []
        for _, row in block.iterrows():
            if len(accepted) >= MAX_SEQLETS_PER_ENHANCER:
                break
            if any(row["start_seq"] < a["end_seq"] and a["start_seq"] < row["end_seq"]
                   for a in accepted):
                continue
            accepted.append(row)
        chosen_for_gene.extend(accepted)

        if len(chosen_for_gene) >= MAX_SEQLETS_PER_GENE:
            break

    kept.extend(chosen_for_gene[:MAX_SEQLETS_PER_GENE])

selected_seqlets = pd.DataFrame(kept).reset_index(drop=True)

selected_seqlets

In [ ]:
# Report what the rules actually did.
per_gene = selected_seqlets.groupby("gene").agg(
    n_seqlets=("label", "size"),
    n_enhancers=("enh_key", "nunique"),
    n_eligible=("tf_eligible", "sum"),
)
per_gene["gene_cap_bound"] = per_gene["n_seqlets"] >= MAX_SEQLETS_PER_GENE
print(f"selected {len(selected_seqlets)} seqlets across {len(per_gene)} genes")
print(f"seqlets per gene: median={per_gene.n_seqlets.median():.0f}  "
      f"range=[{per_gene.n_seqlets.min()},{per_gene.n_seqlets.max()}]")
print(f"genes where the per-gene cap bound: {int(per_gene.gene_cap_bound.sum())}")
print(f"eligible fraction among selected: "
      f"{selected_seqlets.tf_eligible.mean():.1%}")

# Genes that lost an enhancer entirely contribute fewer cross-enhancer pairs.
lost = per_gene[per_gene.n_enhancers < gene_windows.loc[per_gene.index, "n_enhancers"]]
if not lost.empty:
    print(f"[info] {len(lost)} gene(s) have enhancers with no selected seqlet:")
    print(lost.head(10).to_string())


In [ ]:
# ---------------------------------------------------------------------------
# 6. Budget dry run. Nothing is submitted before this number is read.
# ---------------------------------------------------------------------------

def enumerate_gene_combinations(labels: list[str],
                                include_within: bool = True) -> list[tuple]:
    """Baseline, singletons, pairs, and the all-scrambled state.

    Restricting to pairs keeps the cost O(N^2) rather than O(2^N). Pairs are
    classified downstream as within-enhancer or across-enhancer by comparing
    the "GENE_enhN" prefix of the two labels.

    Args:
        labels: Seqlet labels for one gene.
        include_within: Keep pairs whose members share an enhancer. Setting
            this False saves budget but removes the controlled comparison
            between within-enhancer and cross-enhancer interaction.

    Returns:
        Sorted list of label tuples, always starting with the empty tuple.
    """
    combos = [()]
    combos += [(label,) for label in labels]
    for a, b in itertools.combinations(labels, 2):
        same_enhancer = a.split("|")[0] == b.split("|")[0]
        if same_enhancer and not include_within:
            continue
        combos.append((a, b))
    if len(labels) > 2:
        combos.append(tuple(labels))
    return sorted(set(combos), key=lambda c: (len(c), c))


budget_rows = []
for gene, group in selected_seqlets.groupby("gene"):
    labels = group["label"].tolist()
    combos = enumerate_gene_combinations(labels, INCLUDE_WITHIN_ENHANCER_PAIRS)
    n_within = sum(1 for c in combos
                   if len(c) == 2 and c[0].split("|")[0] == c[1].split("|")[0])
    n_across = sum(1 for c in combos
                   if len(c) == 2 and c[0].split("|")[0] != c[1].split("|")[0])
    budget_rows.append(dict(
        gene=gene, n_seqlets=len(labels), n_combos=len(combos),
        n_within_pairs=n_within, n_across_pairs=n_across,
        # One deterministic baseline, everything else replicated.
        n_sequences=1 + (len(combos) - 1) * N_SHUFFLES,
    ))

budget = pd.DataFrame(budget_rows)
print(budget.to_string(index=False))
print(f"\nTOTAL sequences: {budget.n_sequences.sum():,}")
print(f"  across-enhancer pairs: {budget.n_across_pairs.sum():,}")
print(f"  within-enhancer pairs: {budget.n_within_pairs.sum():,}")

In [ ]:
# ---------------------------------------------------------------------------
# 7. Gene-level combination generator
#
# For each gene and each combination, produce ONE modified sequence per
# enhancer of that gene. Scrambled windows are spliced from that enhancer's own
# donor shuffle at index k, and the SAME k is used for every enhancer in the
# combination so that the paired-replicate structure survives across enhancers.
# ---------------------------------------------------------------------------

_IDX_TO_NUC = np.array(["A", "C", "G", "T"])


def _onehot_to_seq(onehot: torch.Tensor) -> str:
    """Convert a [4, L] one-hot tensor to an uppercase nucleotide string."""
    return "".join(_IDX_TO_NUC[onehot.argmax(dim=0).numpy()])


def generate_gene_combination_df(
    seqlets: pd.DataFrame,
    gene_enhancers: pd.DataFrame,
    dishuff: torch.Tensor,
    n_shuffles: int = N_SHUFFLES,
    include_within: bool = INCLUDE_WITHIN_ENHANCER_PAIRS,
) -> pd.DataFrame:
    """Enumerate every scramble combination per gene.

    Args:
        seqlets: Selected seqlets with `gene`, `enh_key`, `label`, `seq_idx`,
            `start_seq`, `end_seq`.
        gene_enhancers: Gene-qualified enhancer table with reference sequences.
        dishuff: Donor shuffles, shape (n_enhancers, n_shuffles, 4, 313).
        n_shuffles: Replicates per non-trivial combination.
        include_within: Whether to retain within-enhancer pairs.

    Returns:
        One row per sequence, carrying the modified sequence of every enhancer
        of the gene as an aligned list.
    """
    rows = []

    for gene, group in seqlets.groupby("gene"):
        # Enhancers of this gene, in a fixed order. This order defines the
        # alignment of `enh_sequences`, `ATAC_counts` and `target_enhancer`.
        enh_table = (gene_enhancers[gene_enhancers.gene == gene]
                     .drop_duplicates("enh_key")
                     .sort_values("start")
                     .reset_index(drop=True))
        enh_keys = enh_table["enh_key"].tolist()
        reference = {r.enh_key: r.seq for r in enh_table.itertuples()}
        seq_idx_of = {r.enh_key: int(r.seq_idx) for r in enh_table.itertuples()}

        # Pre-render the donor shuffles once per enhancer.
        donors = {
            key: [_onehot_to_seq(dishuff[seq_idx_of[key]][k])
                  for k in range(n_shuffles)]
            for key in enh_keys
        }

        # Windows to splice, keyed by label.
        windows = {r.label: (r.enh_key, int(r.start_seq), int(r.end_seq))
                   for r in group.itertuples()}

        combos = enumerate_gene_combinations(list(windows), include_within)

        for combo_id, combo in enumerate(combos):
            # The unperturbed reference is deterministic and needs no replicates.
            shuffle_indices = [None] if not combo else range(n_shuffles)

            for k in shuffle_indices:
                sequences = dict(reference)   # copy per row
                for label in combo:
                    enh_key, start, end = windows[label]
                    donor = donors[enh_key][k]
                    seq = list(sequences[enh_key])
                    seq[start:end] = list(donor[start:end])
                    sequences[enh_key] = "".join(seq)

                enhancers_hit = {label.split("|")[0] for label in combo}
                if len(combo) == 2:
                    pair_type = "within" if len(enhancers_hit) == 1 else "across"
                elif len(combo) == 1:
                    pair_type = "single"
                elif len(combo) == 0:
                    pair_type = "baseline"
                else:
                    pair_type = "all"

                rows.append(dict(
                    name=gene,                       # matches enhancer-level schema
                    combo_id=combo_id,
                    combo_label="scr[" + ",".join(combo) + "]",
                    scrambled_seqlets=list(combo),
                    unscrambled_seqlets=[l for l in windows if l not in combo],
                    n_scrambled=len(combo),
                    n_enhancers_hit=len(enhancers_hit),
                    pair_type=pair_type,
                    shuffle_idx=np.nan if k is None else k,
                    enh_keys=enh_keys,
                    enh_sequences=[sequences[key] for key in enh_keys],
                ))

    return pd.DataFrame(rows)


cross_df = generate_gene_combination_df(selected_seqlets, gene_enhancers, dishuff)

print(f"rows: {len(cross_df):,}   genes: {cross_df.name.nunique()}")
print(cross_df.pair_type.value_counts().to_string())

# Every modified sequence must still be exactly one enhancer long.
lengths = {len(s) for row in cross_df.enh_sequences for s in row}
assert lengths == {ENH_LEN}, f"unexpected sequence lengths: {lengths}"

# The baseline row must be identical to the reference for every enhancer.
for gene, group in cross_df.groupby("name"):
    base = group[group.n_scrambled == 0].iloc[0]
    ref = (gene_enhancers[gene_enhancers.gene == gene]
           .drop_duplicates("enh_key").sort_values("start"))
    assert base["enh_sequences"] == ref["seq"].tolist(), \
        f"baseline sequence altered for {gene}"
print("baseline rows match the reference sequences.")

In [ ]:
# ---------------------------------------------------------------------------
# 8. Gene-level model input builder
#
# The 1 Mb reference window is fetched once per gene and cached. Each row then
# splices its modified enhancer sequences into a copy at the correct offsets.
# Building in blocks keeps peak memory bounded, since each input string is
# about 1 MB.
# ---------------------------------------------------------------------------

def build_gene_window(gene: str, hg38, gene_windows: pd.DataFrame) -> str:
    """Fetch the reference 1 Mb window for a gene, N-padded at contig ends."""
    info = gene_windows.loc[gene]
    chrom = info["chrom"]
    win_start, win_end = int(info["win_start"]), int(info["win_end"])

    chrom_len = len(hg38[chrom])
    left_pad = max(0, -win_start)
    right_pad = max(0, win_end - chrom_len)
    fetch_start = max(0, win_start)
    fetch_end = min(chrom_len, win_end)

    core = str(hg38[chrom][fetch_start:fetch_end]).upper()
    window = "N" * left_pad + core + "N" * right_pad
    assert len(window) == SEQ_LEN, f"window length {len(window)} != {SEQ_LEN}"
    return window


def enhancer_offsets(gene: str, enh_keys: list[str],
                     gene_enhancers: pd.DataFrame,
                     gene_windows: pd.DataFrame) -> list[int]:
    """Offset of each enhancer inside the gene window, in the given order."""
    win_start = int(gene_windows.loc[gene, "win_start"])
    lookup = (gene_enhancers[gene_enhancers.gene == gene]
              .drop_duplicates("enh_key").set_index("enh_key")["start"])
    offsets = [int(lookup[key]) - win_start for key in enh_keys]
    # An enhancer falling outside the window would be silently truncated.
    for key, off in zip(enh_keys, offsets):
        assert 0 <= off and off + ENH_LEN <= SEQ_LEN, \
            f"{key} lies outside the window for {gene} (offset {off})"
    return offsets


def make_gene_model_inputs(block: pd.DataFrame, window: str,
                           offsets: list[int]) -> list[str]:
    """Splice each row's modified enhancers into a copy of the gene window."""
    inputs = []
    for _, row in block.iterrows():
        seq = list(window)
        for offset, enh_seq in zip(offsets, row["enh_sequences"]):
            seq[offset:offset + ENH_LEN] = list(enh_seq)
        inputs.append("".join(seq))
    return inputs

In [ ]:
# ---------------------------------------------------------------------------
# 9. Prediction driver
#
# Readouts, deliberately matching the enhancer-level schema:
#   RNA_counts  : list, one value per target gene of the window
#   ATAC_counts : list, one value per enhancer of the gene
#   target_enhancer : the aligned enhancer key list
#
# With that schema, element_epistasis.enhancer_analysis works on the output unchanged.
# ---------------------------------------------------------------------------

def predict_cross_enhancer_effects(
    combinations_df: pd.DataFrame,
    gene_enhancers: pd.DataFrame,
    gene_windows: pd.DataFrame,
    hg38,
    model,
    curies,
    seq_len: int = SEQ_LEN,
    input_block: int = INPUT_BLOCK,
):
    """Predict RNA and per-enhancer ATAC for every combination, gene by gene."""
    raw_frames, agg_frames = [], []
    genes = combinations_df["name"].unique()

    for gene_nr, gene in enumerate(genes, start=1):
        block_all = combinations_df[combinations_df.name == gene].copy()
        enh_keys = block_all["enh_keys"].iloc[0]
        print(f"[{gene_nr}/{len(genes)}] {gene}: {len(block_all)} sequences, "
              f"{len(enh_keys)} enhancers", flush=True)

        window = build_gene_window(gene, hg38, gene_windows)
        offsets = enhancer_offsets(gene, enh_keys, gene_enhancers, gene_windows)

        # Sanity: the baseline input must reproduce the reference genome at
        # every enhancer offset. If this fails the coordinate mapping is wrong.
        baseline_row = block_all[block_all.n_scrambled == 0].iloc[[0]]
        check_input = make_gene_model_inputs(baseline_row, window, offsets)[0]
        for key, offset, ref_seq in zip(enh_keys, offsets,
                                        baseline_row["enh_sequences"].iloc[0]):
            assert check_input[offset:offset + ENH_LEN] == ref_seq, \
                f"embedding mismatch for {key} in {gene}"

        interval = genome.Interval(
            chromosome=str(gene_windows.loc[gene, "chrom"]),
            start=int(gene_windows.loc[gene, "win_start"]),
            end=int(gene_windows.loc[gene, "win_end"]),
        )

        gene_mask = annot = None
        atac_all, rna_all, target_all = [], [], []

        for begin in range(0, len(block_all), input_block):
            block = block_all.iloc[begin:begin + input_block]
            inputs = make_gene_model_inputs(block, window, offsets)

            out_pred = tc.predict_with_backoff(
                model=model,
                sequences=inputs,
                interval=interval,
                requested_outputs=[dna_client.OutputType["ATAC"],
                                   dna_client.OutputType["RNA_SEQ"]],
                ontology_terms=curies,
                chunk_size=64,
                max_workers=6,
            )
            del inputs

            # Gene masks depend only on the interval, so extract them once.
            if gene_mask is None:
                first = next(p for p in out_pred if p is not None)
                gene_mask, annot = exon_extractor.extract(
                    interval=first.rna_seq.interval)
                is_target = annot.gene_name == gene
                annot = annot.loc[is_target]
                gene_mask = gene_mask[:, is_target.values]
                if annot.empty:
                    print(f"  [warn] {gene} has no exon mask in this window; "
                          f"RNA will be empty")

            for seq_out in out_pred:
                if seq_out is None:
                    # A failed chunk. Recorded as missing rather than dropped,
                    # so row alignment with block_all is preserved.
                    atac_all.append([np.nan] * len(enh_keys))
                    rna_all.append([np.nan])
                    target_all.append([gene])
                    continue

                atac_track = seq_out.atac.values.mean(axis=1)
                atac_all.append([float(atac_track[off:off + ENH_LEN].sum())
                                 for off in offsets])

                counts, targets = [], []
                for mask_idx, row in enumerate(annot.itertuples()):
                    stranded = seq_out.rna_seq.metadata.strand.isin(
                        [row.strand, "."])
                    value = (seq_out.rna_seq.values[:, stranded].mean(axis=1)
                             [gene_mask[:, mask_idx]].mean())
                    counts.append(float(value))
                    targets.append(row.gene_name)
                rna_all.append(counts if counts else [np.nan])
                target_all.append(targets if targets else [gene])

            del out_pred

        block_all["ATAC_counts"] = atac_all
        block_all["RNA_counts"] = rna_all
        block_all["target_gene"] = target_all
        block_all["target_enhancer"] = [enh_keys] * len(block_all)

        # The heavy sequence columns are not needed downstream.
        block_all = block_all.drop(columns=["enh_sequences"])
        raw_frames.append(block_all.copy())

        # Aggregate over shuffles, mirroring the within-enhancer pipeline.
        agg = (block_all
               .groupby(["name", "combo_label", "n_scrambled", "pair_type"],
                        observed=True)
               .agg(ATAC_counts=("ATAC_counts",
                                 lambda s: np.nanmean(np.vstack(s), axis=0).tolist()),
                    RNA_counts=("RNA_counts",
                                lambda s: np.nanmean(np.vstack(s), axis=0).tolist()),
                    scrambled_seqlets=("scrambled_seqlets", "first"),
                    unscrambled_seqlets=("unscrambled_seqlets", "first"),
                    target_enhancer=("target_enhancer", "first"),
                    target_gene=("target_gene", "first"),
                    n_enhancers_hit=("n_enhancers_hit", "first"))
               .reset_index())
        agg_frames.append(agg)

    return raw_frames, agg_frames

In [ ]:
if RUN_PREDICTIONS:
    # ---------------------------------------------------------------------------
    # 10. Run and save
    #
    # `shuffle_idx` is retained in the raw frame. Every paired statistic downstream
    # depends on it, and the aggregated frame alone cannot reconstruct it.
    # ---------------------------------------------------------------------------

    raw_frames, agg_frames = predict_cross_enhancer_effects(
        cross_df, gene_enhancers, gene_windows, hg38, model, K562_curie)

    cross_scram_effects_raw = pd.concat(raw_frames).reset_index(drop=True)
    cross_scram_effects = pd.concat(agg_frames).reset_index(drop=True)


In [ ]:
if RUN_PREDICTIONS:
    OUT_DIR = paths.PROCESSED
    cross_scram_effects_raw.to_parquet(f"{OUT_DIR}/TF_cross_enhancer_raw.parquet")
    cross_scram_effects.to_parquet(f"{OUT_DIR}/TF_cross_enhancer.parquet")

    print(f"raw: {cross_scram_effects_raw.shape}   agg: {cross_scram_effects.shape}")
    print(f"missing predictions: "
          f"{int(cross_scram_effects_raw.ATAC_counts.apply(lambda v: np.isnan(v).any()).sum())}"
          f" of {len(cross_scram_effects_raw)} rows")
    print(cross_scram_effects_raw.pair_type.value_counts().to_string())


In [ ]:
if RUN_PREDICTIONS:
    # ---------------------------------------------------------------------------
    # 11. Schema check
    #
    # Confirms the output is shaped like the enhancer-level scram_effects, so that
    # element_epistasis.enhancer_analysis applies without modification.
    # ---------------------------------------------------------------------------
    required = {"name", "n_scrambled", "scrambled_seqlets", "shuffle_idx",
                "RNA_counts", "ATAC_counts", "target_enhancer"}
    missing_cols = required - set(cross_scram_effects_raw.columns)
    assert not missing_cols, f"schema incomplete: {missing_cols}"

    # ATAC_counts must align with target_enhancer element for element.
    bad = cross_scram_effects_raw.apply(
        lambda r: len(r["ATAC_counts"]) != len(r["target_enhancer"]), axis=1)
    assert not bad.any(), f"{int(bad.sum())} rows have misaligned ATAC vectors"

    print("Schema matches the enhancer-level tables. Ready for:")
    print("  ea.expand_locus_values(cross_scram_effects_raw, 'ATAC_counts')")
    print("  ea.build_pairwise_epistasis(...)  /  ea.build_coop_table(...)")
    print("  ea.build_interdependency(...)")


## Cross enhancer TF analysis

In [ ]:
import element_epistasis.enhancer_analysis as ea
import element_epistasis.enhancer_plots as ep
import element_epistasis.glm as glm
import element_epistasis.GLM_fit as glmp
import element_epistasis.cross_plots as cp
import element_epistasis.cross_level as cl


import importlib
importlib.reload(ea)
importlib.reload(cp)
importlib.reload(ep)
importlib.reload(glm)

In [ ]:
OUT_DIR = paths.PROCESSED
cross_scram_effects_raw = pd.read_parquet(f"{OUT_DIR}/TF_cross_enhancer_raw.parquet")
cross_scram_effects = pd.read_parquet(f"{OUT_DIR}/TF_cross_enhancer.parquet")

# As above: the seqlet selection is recomputed, so confirm it matches the run.
_stored = {lab for labs in cross_scram_effects_raw["scrambled_seqlets"] for lab in labs}
_missing = _stored - set(selected_seqlets["label"])
assert not _missing, f"stored cross-enhancer labels not reproduced: {sorted(_missing)[:10]}"


In [ ]:
invitro_betas = pd.read_csv(paths.RAW / "pair_interaction_adj.csv")

measured_pairs = cl.normalise_measured_pairs(
    invitro_betas,
    collapse="mean",      # or "mean", or "none" to keep every record
)

In [ ]:
enh_inters = pd.read_csv(paths.PROCESSED / "enhancer_insilico_KO_inters.csv")

In [ ]:
enh_inters.head(3)

In [ ]:
#flatten RNA counts list
cross_scram_effects_raw["RNA_counts"] = [l[0] for l in cross_scram_effects_raw.RNA_counts]

In [ ]:
#fit the rna
cross_glm_res_rna = glm.fit_all_genes(
    df_raw=cross_scram_effects_raw,
    name_col="name",
    response_col="RNA_counts",
    family="gamma",
    max_order=2,
    round_nb=False,  # Set True if using family="nb" with continuous predictions
)

cross_beta_rna = glm.collect_terms(cross_glm_res_rna, interactions_only=False)
cross_beta_rna = tc.add_bh_correction(cross_beta_rna)

cross_inter_rna = glm.classify_interaction_terms(
    cross_beta_rna,
    term_col="term",
    gene_col="gene",
    unit_separator="|",  # Separator between enhancer and motif name
)

cross_inter_rna = glm.attach_motif_distance(cross_inter_rna, selected_seqlets)

#fir the atac
cross_scram_effects_atac_long     = ea.expand_locus_values(cross_scram_effects_raw, "ATAC_counts")
cross_glm_res_atac = glm.fit_all_loci(
    long_df=cross_scram_effects_atac_long,
    response_col="ATAC_counts",
    gene_col="name",
    locus_col="locus",
    family="gamma",
    max_order=2,
)

cross_beta_atac = glm.collect_terms(cross_glm_res_atac)
cross_inter_atac = glm.classify_interaction_terms(cross_beta_atac)

interdep = tc.add_bh_correction(glm.build_motif_interdependency(cross_beta_atac))
pair_agg_rna = glm.aggregate_pairs_to_enhancer(cross_inter_rna)          # cross-level join key
scope_rna = glm.summarise_scope_comparison(cross_inter_rna)      

In [ ]:
sidx = [(seq_idx, name) for seq_idx, name in enumerate(names) if "SLC25A37" in name.upper()]

fig, ax = plt.subplots(len(sidx), 1, figsize=(20, 9))
for a, (idx, name) in enumerate(sidx):
    ism_res = torch.tensor(ism_enhancer[names[idx]]["ATAC"]).permute(1, 0)
    plot_logo(ism_res, ax = ax[a],
          annotations=trimmed_seqlets.loc[trimmed_seqlets.seq_idx == idx, ["example_idx", "start","end", "attribution"]],
          score_key='attribution', n_tracks=5,)
    ax[a].set_ylabel(f"{name}", rotation = 90,)
plt.tight_layout()
plt.show()    

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(26, 4))
glmp.plot_coef_forest(cross_glm_res_rna["SLC25A37"], ax=ax[0])
glmp.plot_coef_forest(cross_glm_res_atac['SLC25A37@SLC25A37_enh1'], ax=ax[1])
glmp.plot_coef_forest(cross_glm_res_atac['SLC25A37@SLC25A37_enh2'], ax=ax[2])

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(26, 4))
glmp.plot_coef_forest(cross_glm_res_rna["CXXC5"], ax=ax[0])
glmp.plot_coef_forest(cross_glm_res_atac['CXXC5@CXXC5_enh1'], ax=ax[1])
glmp.plot_coef_forest(cross_glm_res_atac['CXXC5@CXXC5_enh2'], ax=ax[2])

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(26, 6))
glmp.plot_coef_forest(cross_glm_res_rna["FAM89A"], ax=ax[0])
glmp.plot_coef_forest(cross_glm_res_atac['FAM89A@FAM89A_enh1'], ax=ax[1])
glmp.plot_coef_forest(cross_glm_res_atac['FAM89A@FAM89A_enh3'], ax=ax[2])

plt.show()

In [ ]:
importlib.reload(cp)

In [ ]:
cp.make_main_figure(cross_inter_rna, interdep, pair_agg=pair_agg_rna, enh_inters=enh_inters,
                    measured_pairs=measured_pairs.rename(columns = {"interaction_beta": "coef"}))
plt.show()

In [ ]:
importlib.reload(cp)

In [ ]:
cp.make_grammar_figure(cross_inter_rna, )
plt.show()

In [ ]:
fig, axes = plt.subplots(1,1,figsize=(18,18))
cp.plot_motif_interdependency(interdep,
                              #genes=["FAM89A", "CXXC5", "MGAT1"],
                              show_only_sig=True,
                               hatch_nonsignificant = True, 
                               ylabel_pad = 53.0,
                               gene_label_pad = 54.0,
                              ax = axes)
plt.show()

In [ ]:
fig, axes = plt.subplots(1,1,figsize=(6,6))
cp.plot_pair_interdependency(interdep, #genes=["FAM89A", "CXXC5", "MGAT1"],
                             ax = axes)
plt.show()


In [ ]:
linked_pairs = cl.link_motif_pairs_to_enhancers(
    pair_agg_rna,                 # glm.aggregate_pairs_to_enhancer(inters_rna)
    enh_inters,               # ea.split_coef_table(enh_coef)[1]
    measured_pairs,           # step 1; omit to get the in silico arm alone
)
pair_link = cl.summarise_pair_link(linked_pairs)

In [ ]:
pair_link 

In [ ]:
print(len(linked_pairs),
      linked_pairs["enh_coef"].notna().sum(),
      linked_pairs["interaction_beta"].notna().sum())

In [ ]:
within_gene = cl.summarise_within_gene_link(
    linked_pairs, x_col="motif_mean", y_col="enh_coef", gene_col="gene",
)
within_gene

In [ ]:
within_gene_measured = cl.summarise_within_gene_link(
    linked_pairs, x_col="motif_mean", y_col="interaction_beta", gene_col="gene",
)
within_gene_measured

In [ ]:
import importlib
import element_epistasis.cross_report
importlib.reload(element_epistasis.cross_report)
import element_epistasis.cross_report as cr

In [ ]:
cross_summary = cr.write_cross_summary(
    paths.SUMMARIES / "cross_enhancer_summary.txt",
    inters_rna=cross_inter_rna,
    beta_rna=cross_beta_rna,
    inters_atac=cross_inter_atac,
    beta_atac=cross_beta_atac,
    interdep=interdep,
    pair_agg=pair_agg_rna,
    #scope=scope_rna,
    enh_inters=enh_inters,
    enh_mains = enh_mains,
    tf_pairs=tc.add_bh_correction(tc.build_pairwise_interactions(scram_effects_raw, "ATAC_counts", seqlets=fulls_seqlets)),              # earlier within-enhancer run, if to hand
    glm_results_rna=cross_glm_res_rna,
    glm_results_atac=cross_glm_res_atac,
    top_n = 15,
    exemplar_n = 10,
)